# 12 · Prefix tuning — trainable K/V at *every* layer

**Recap (11 & 09):** prompt tuning added trainable vectors at the **input only**.
Attention at each layer works through **Keys and Values**. **Prefix tuning** prepends
trainable **K** and **V** vectors directly into **every layer's** attention — not just
the input. More places to steer → more leverage than prompt tuning.

This is also the **last mechanism**, so we close with a real parameter comparison of
*all* the PEFT methods on the actual Qwen2.5-Coder-1.5B.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## Step 1 — The mechanism: virtual K/V injected at each layer

Attention blends the **Values**, weighted by Query–**Key** matches. Prefix tuning adds
`P` extra trainable Keys and Values at the front of *every* layer's attention. Real
tokens can now attend to this learned "virtual context" at all depths — not just at
the input like prompt tuning.

In [ ]:
d, P, layers = 4, 2, 3

# prompt tuning: vectors at the INPUT only (one place)
prompt_params = P * d

# prefix tuning: a trainable K and V at EVERY layer
prefix_K = rng.normal(size=(layers, P, d))
prefix_V = rng.normal(size=(layers, P, d))
prefix_params = prefix_K.size + prefix_V.size

print("prompt tuning trainable :", prompt_params, " (input only)")
print("prefix tuning trainable :", prefix_params, f" (K & V across {layers} layers)")

Even on this toy 3-layer model, prefix tuning trains more than prompt tuning — because
it pays at *every* layer. Now let's see it at real scale.

## Step 2 — All PEFT methods, counted on Qwen2.5-Coder-1.5B

Here are the real config numbers for our base model — including the **grouped-query
attention** detail: 12 query heads but only **2 key/value heads**, so K and V are just
`2 × 128 = 256`-wide, not the full 1536. That shrinks anything touching K/V.

In [ ]:
# --- real Qwen2.5-Coder-1.5B-Instruct config ---
hidden  = 1536          # hidden_size
layers  = 28            # num_hidden_layers
ffn     = 8960          # intermediate_size (FFN hidden)
heads, kv_heads = 12, 2 # GQA: 12 query heads, 2 key/value heads
head_dim = hidden // heads      # 128
q_dim   = heads * head_dim      # 1536  (query / output width)
kv_dim  = kv_heads * head_dim   # 256   (key/value width, shrunk by GQA)
total   = 1.54e9                # ~total parameters

P = 20      # virtual tokens (prompt / prefix)
r = 16      # LoRA rank

trainable = {
    # LoRA on q_proj (1536->1536) and v_proj (1536->256 under GQA), every layer
    "LoRA (r=16, q&v)": (r*(hidden+q_dim) + r*(hidden+kv_dim)) * layers,
    # BitFit: Qwen has bias only on q/k/v projections (RMSNorm has no bias)
    "BitFit (biases)":  (q_dim + kv_dim + kv_dim) * layers,
    # IA3: multipliers on K, V (256 each) and FFN (8960), every layer
    "IA3 (K,V,FFN)":    (kv_dim + kv_dim + ffn) * layers,
    # Prompt tuning: P soft vectors at the input only
    "Prompt tuning":    P * hidden,
    # Prefix tuning: P virtual K and V at EVERY layer
    "Prefix tuning":    P * layers * 2 * hidden,
}

print(f"{'method':<20}{'trainable params':>18}{'% of 1.54B':>13}")
print("-" * 51)
for name, n in sorted(trainable.items(), key=lambda kv: kv[1]):
    print(f"{name:<20}{n:>18,}{n/total:>12.4%}")

## Step 3 — What the numbers tell us

- **Prompt tuning is the smallest** (~31k params) — and recall notebook 11 warned it's
  also the *weakest* at this model size. Cheapest, least leverage.
- **BitFit is tiny too** (~57k) — surprisingly small, because Qwen uses **RMSNorm
  (no bias)**, so only the q/k/v projections carry biases. On an architecture with more
  biases, BitFit would be bigger.
- **IA3** (~265k) — a few hundred-thousand multipliers across K/V/FFN.
- **Prefix tuning** (~1.7M) — *far* bigger than prompt tuning even though both use
  `P = 20` virtual tokens, because prefix injects at **every one of the 28 layers**
  while prompt only injects at the input. That extra reach is exactly the leverage you
  pay for.
- **LoRA (r=16)** (~2.2M) — the largest here, yet still just **0.14%** of the model
  (and you can shrink it by lowering `r`).

The headline: **every method trains well under 0.2%** of the 1.54B parameters — that's
the "parameter-efficient" in PEFT. Yet they span ~**70×** among themselves (prompt 31k
→ LoRA 2.2M), trading parameter budget for expressive power.

**Honest caveats:**
- These are **computed from the config**, not read off the loaded model. The ground
  truth is `count_trainable(model)` in `model_factory.py` during the real GPU run.
- Prefix uses the PEFT default of full hidden-width K/V per layer; in principle GQA
  could reduce it, but the implementation doesn't.

## Recap — the whole mechanism axis

| family | method | trains | ~params on Qwen-1.5B |
|---|---|---|---|
| reparameterization | LoRA / QLoRA | low-rank `B·A` on weights | ~2.2M (0.14%) |
| selective | BitFit | existing bias terms | ~57k (0.004%) |
| additive | IA3 | multipliers on K/V/FFN | ~265k (0.017%) |
| additive | prompt tuning | soft vectors at input | ~31k (0.002%) |
| additive | prefix tuning | virtual K/V at every layer | ~1.7M (0.11%) |

All freeze the bulk of the model and train a sliver — they differ in **what** that
sliver is, **where** it lives, and **how big** it is. That's the mechanism axis the lab
compares, all scored by the same execution-accuracy harness.

**Triple BAM!** You've covered every PEFT mechanism. Next: the *outcome* evidence — run
them on Kaggle and plot trainable-% vs accuracy — or the **objective** axis (DPO ✅,
PPO next): changing *what* you train on instead of *how many* parameters.